In [1]:
import pandas as pd
import numpy as np
import random
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory
import yfinance as yf
from pathlib import Path
import os
import matplotlib.pyplot as plt


In [2]:
anos = ['2025-12-31',
 '2024-12-31',
 '2023-12-31',
 '2022-12-31',
 '2021-12-31',
 '2020-12-31',
 '2019-12-31',
 '2018-12-31',
 '2017-12-31',
 '2016-12-31',
 '2015-12-31',
 ]

apenas_ano = []
for an in anos:
    ano = an.split("-")[0]
    apenas_ano.append(ano)
    # print(ano, type(ano))
    os.makedirs(f'score_mf/{ano}', exist_ok=True)
apenas_ano

['2025',
 '2024',
 '2023',
 '2022',
 '2021',
 '2020',
 '2019',
 '2018',
 '2017',
 '2016',
 '2015']

In [3]:
basedados_ativos = Path('../../base_dados/brapi/retornos/retornos.csv')
basedados_ibov = Path('../../base_dados/retorno_ibov_2015_2026.csv') 

lista_magic_formula = []

for filename in os.listdir(path='../../base_dados/brapi/magic_formula/'):
    
    # 2. Reconstruct the full absolute or relative path to the file
    full_path = os.path.join('../../base_dados/brapi/magic_formula/', filename)
    
    # 3. Check if the current item is actually a file (and not a subfolder)
    # if os.path.isfile(full_path):
        
    #     # 4. Open and process the file safely
    #     with open(full_path, "r", encoding="utf-8") as file:
    #         content = file.read()
    #         print(f"--- Content of {filename} ---")
    #         print(pd.read_csv(full_path))
    dicio = {
        'ativo':filename,
        'data':pd.read_csv(full_path)
    }

    lista_magic_formula.append(dicio)

df_ativos=pd.read_csv(basedados_ativos).set_index(['date']).fillna(0)

df_ibov=pd.read_csv(basedados_ibov).set_index(['Date']).fillna(0)



In [4]:
print(df_ativos.shape[0])
print(df_ibov.shape)


2863
(2849, 1)


In [5]:
lista_magic_formula

[{'ativo': 'ABEV3',
  'data':     Unnamed: 0     endDate       ROC        EY
  0            0  2025-12-31  0.903659  0.111531
  1            1  2024-12-31  0.624135  0.141094
  2            2  2023-12-31  0.840339  0.101218
  3            3  2022-12-31  0.646090  0.093623
  4            4  2021-12-31  0.585286  0.089962
  5            5  2020-12-31  0.598088  0.085787
  6            6  2019-12-31  0.637351  0.073615
  7            7  2018-12-31  0.824850  0.092799
  8            8  2017-12-31  1.105830  0.069359
  9            9  2016-12-31  1.198593  0.096538
  10          10  2015-12-31  1.084847  0.104877
  11          11  2014-12-31  1.081966  0.102054
  12          12  2013-12-31  0.892491  0.099119
  13          13  2012-12-31  0.762046  0.000204},
 {'ativo': 'ALOS3',
  'data':     Unnamed: 0     endDate       ROC        EY
  0            0  2025-12-31  0.719754  0.069923
  1            1  2024-12-31  0.690710  0.089326
  2            2  2023-12-31  3.051214  0.259800
  3        

In [6]:
dict_df = {}
for i in range(len(apenas_ano)):
    # print(apenas_ano[i])
    df = df_ativos[df_ativos.index.str.startswith(apenas_ano[i])]
    df = df.drop(columns=df.columns[(df == 0).all()])

    print(apenas_ano[i],"---- Quantidade de ativos",len(df.columns))
    dict_df[apenas_ano[i]] = df

2025 ---- Quantidade de ativos 78
2024 ---- Quantidade de ativos 78
2023 ---- Quantidade de ativos 78
2022 ---- Quantidade de ativos 77
2021 ---- Quantidade de ativos 77
2020 ---- Quantidade de ativos 73
2019 ---- Quantidade de ativos 72
2018 ---- Quantidade de ativos 70
2017 ---- Quantidade de ativos 70
2016 ---- Quantidade de ativos 64
2015 ---- Quantidade de ativos 62


In [7]:
[d for d in lista_magic_formula if d['ativo']=='SMTO3']

[{'ativo': 'SMTO3',
  'data':     Unnamed: 0     endDate       ROC        EY
  0            0  2025-03-31  0.099985  0.082003
  1            1  2024-03-31  0.178155  0.132863
  2            2  2023-03-31  0.152647  0.118560
  3            3  2022-03-31  0.182311  0.111522
  4            4  2021-03-31  0.211680  0.104919
  5            5  2020-03-31  0.127322  0.117590
  6            6  2019-03-31  0.091368  0.078677
  7            7  2018-03-31  0.115618  0.093825
  8            8  2017-03-31  0.115688  0.081751
  9            9  2016-03-31  0.122519  0.103271
  10          10  2015-03-31  0.122647  0.113417
  11          11  2014-03-31  0.084475  0.082053
  12          12  2013-03-31  0.053618  0.059590}]

In [10]:
for ano in apenas_ano:
    ativos_do_ano = dict_df[ano].columns          # já filtrados, sem os zerados

    linha = {}
    for j in lista_magic_formula:
        ativo = j['ativo'].replace('.csv', '')
        if ativo not in ativos_do_ano:
            continue

        df_ativo = j['data'].set_index(['endDate']).drop(columns=['Unnamed: 0'])
        # print(ativo,"-",ano)
        # col = next((c for c in (j['ativo'], ativo) if c in df_ativo.columns),
        #            df_ativo.columns[-1])
        # print(col)
        # print(df_ativo)

        alvo = [df_ativo.loc[d].values for d in df_ativo.index if str(d).startswith(ano) and d in anos]
        try:
            roc = alvo[0][0]
            ey = alvo[0][1]
            linha[ativo] = (0.5*roc + 0.5*ey)
            # print(alvo[0][0],alvo[0][1], linha[ativo])
        except Exception as e:
            # print(f"Erro ao acessar alvo para {ativo} no ano {ano}: {e}")
            # print([d for d in lista_magic_formula if d['ativo']==ativo])
            continue
        
    df = pd.DataFrame([linha], index=[ano])
    minimo = df.min(axis=1)
    maximo = df.max(axis=1)
    df_ano = df.sub(minimo, axis=0).div(maximo - minimo, axis=0)

    df_ano.to_csv(f'score_mf/{ano}/mf_{ano}.csv', index_label='ano')

    # print(df_ano)
    # df_ano.to_csv(f'score_mf/{ano}/mf_{ano}.csv', index_label='ano')

    # print(f'{ano}: {len(ativos_do_ano)} negociando, {len(linha)} com score')


# dict_score_todos = []
# dict_score = {}
# for i, ativo in enumerate(df_ativos_2016.columns.tolist()):
#         if ativo == lista_mf_check[i]['ativo']:
#             for ano in anos:
#                 df_temp = lista_mf_check[i]['data'].query("endDate in @ano")
#                 score_anual = (0.5*df_temp['ROC'] + 0.5*df_temp['EY']).sum()
#                 dict_score[(ano, ativo)] = score_anual
#                 print("feito dict do ativo: ",ativo,ano,score_anual)
#         else:
#             print(f" ativo: {ativo}")
#             print(f" ativo_mf: {lista_mf_check[i]['ativo']}")
# df_scores = pd.Series(dict_score).unstack()  # index=anos, columns=ativos
# df_scores = df_scores.replace([np.inf, -np.inf],0)    

# dict_score_todos.append(df_scores)  

In [40]:
df_ativos = df_ativos[df_ativos.index.isin(df_ibov.index)]
print(df_ativos.shape)

(2847, 78)


In [41]:
df_ibov = df_ibov[df_ibov.index.isin(df_ativos[df_ativos.index.isin(df_ibov.index)].index)]
print(df_ibov.shape)

(2847, 1)


In [42]:
df_ativos = df_ativos.filter(lista_ativos_finais)
print(len(df_ativos))

2847


In [43]:
df_ativos

,ABEV3,ALOS3,ANIM3,AXIA3,AZZA3,B3SA3,BBAS3,BBDC3,BBDC4,BBSE3,...,TAEE11,TEND3,TOTS3,UGPA3,USIM5,VALE3,VBBR3,VIVT3,WEGE3,YDUQ3
date,,,,,,,,,,,,,,,,,,,,,
2015-01-05,-0.021863,0.000000,-0.062594,-0.019441,-0.029455,-0.029493,-0.020742,-0.012326,0.002056,-0.031499,...,0.006441,0.000000,-0.030726,-0.024677,-0.060554,-0.015029,0.000000,-0.019833,-0.005811,0.000000
2015-01-06,0.028737,0.000000,-0.141497,0.009026,-0.019570,0.009745,0.013974,0.026667,0.032897,0.038968,...,0.009591,0.000000,-0.054759,-0.006272,0.048899,0.040070,0.000000,-0.024812,-0.015218,0.000000
2015-01-07,0.024826,0.000000,-0.047777,0.035704,0.010184,0.042981,0.044016,0.044159,0.039744,-0.011748,...,-0.005806,0.000000,-0.014933,0.026474,0.059333,0.036695,0.000000,0.034297,-0.001632,0.000000
2015-01-08,0.004024,0.000000,-0.007767,-0.018971,-0.004839,-0.010302,0.003414,0.002209,0.005145,-0.004622,...,0.009555,0.000000,0.022902,-0.003170,-0.050009,0.010628,0.000000,0.040662,0.012846,0.000000
2015-01-09,-0.000610,0.000000,0.003528,-0.021069,-0.010942,-0.021831,-0.043302,-0.035024,-0.043416,-0.028508,...,-0.023140,0.000000,-0.010290,-0.014731,-0.040008,-0.021459,0.000000,-0.015944,-0.005849,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-23,0.015188,0.003801,-0.011494,0.025922,0.036082,0.001360,0.014300,0.007129,0.009050,-0.009319,...,-0.003257,0.017442,0.011009,0.010706,-0.049396,-0.018910,0.015781,0.022064,0.010166,0.009467
2026-06-24,0.000611,0.004083,0.019380,-0.000738,-0.039303,0.016983,-0.006546,-0.010293,-0.010650,0.010713,...,-0.001759,0.033143,-0.005082,-0.006669,0.002309,-0.020786,-0.009527,-0.000877,0.019689,0.029308
2026-06-25,0.000611,0.023658,0.022814,0.016242,0.025375,-0.009411,0.016219,-0.001302,-0.001700,0.004912,...,0.004028,-0.001936,-0.009121,-0.002765,-0.020737,0.011964,0.002061,0.009345,-0.002360,-0.003417


In [44]:
# #ativos
# df_ativos_2015 = df_ativos[df_ativos.index.str.startswith('2015')]
# df_ativos_2016 = df_ativos[df_ativos.index.str.startswith('2016')]
# df_ativos_2017 = df_ativos[df_ativos.index.str.startswith('2017')]
# df_ativos_2018 = df_ativos[df_ativos.index.str.startswith('2018')]
# df_ativos_2019 = df_ativos[df_ativos.index.str.startswith('2019')]
# df_ativos_2020 = df_ativos[df_ativos.index.str.startswith('2020')]
# df_ativos_2021 = df_ativos[df_ativos.index.str.startswith('2021')]
# df_ativos_2022 = df_ativos[df_ativos.index.str.startswith('2022')]
# df_ativos_2023 = df_ativos[df_ativos.index.str.startswith('2023')]
# df_ativos_2024 = df_ativos[df_ativos.index.str.startswith('2024')]
# df_ativos_2025 = df_ativos[df_ativos.index.str.startswith('2025')]
# df_ativos_2026 = df_ativos[df_ativos.index.str.startswith('2026')]
df_ativos_2015 = df_ativos.loc['2015-10-01':'2016-03-31']
df_ativos_2016 = df_ativos.loc['2016-10-01':'2017-03-31']
df_ativos_2017 = df_ativos.loc['2017-10-01':'2018-03-31']
df_ativos_2018 = df_ativos.loc['2018-10-01':'2019-03-31']
df_ativos_2019 = df_ativos.loc['2019-10-01':'2020-03-31']
df_ativos_2020 = df_ativos.loc['2020-10-01':'2021-03-31']
df_ativos_2021 = df_ativos.loc['2021-10-01':'2022-03-31']
df_ativos_2022 = df_ativos.loc['2022-10-01':'2023-03-31']
df_ativos_2023 = df_ativos.loc['2023-10-01':'2024-03-31']   
df_ativos_2024 = df_ativos.loc['2024-10-01':'2025-03-31']
df_ativos_2025 = df_ativos.loc['2025-10-01':'2026-03-31']
# df_ativos_2026 = df_ativos.loc['2026-04-01':'2027-03-31']

df_ativos_2015.to_csv('retornos_ativos_otimizacao/df_ativos_2015.csv')
df_ativos_2016.to_csv('retornos_ativos_otimizacao/df_ativos_2016.csv')
df_ativos_2017.to_csv('retornos_ativos_otimizacao/df_ativos_2017.csv')
df_ativos_2018.to_csv('retornos_ativos_otimizacao/df_ativos_2018.csv')
df_ativos_2019.to_csv('retornos_ativos_otimizacao/df_ativos_2019.csv')
df_ativos_2020.to_csv('retornos_ativos_otimizacao/df_ativos_2020.csv')
df_ativos_2021.to_csv('retornos_ativos_otimizacao/df_ativos_2021.csv')
df_ativos_2022.to_csv('retornos_ativos_otimizacao/df_ativos_2022.csv')
df_ativos_2023.to_csv('retornos_ativos_otimizacao/df_ativos_2023.csv')
df_ativos_2024.to_csv('retornos_ativos_otimizacao/df_ativos_2024.csv')
df_ativos_2025.to_csv('retornos_ativos_otimizacao/df_ativos_2025.csv')
# df_ativos_2026.to_csv('retornos_ativos_otimizacao/df_ativos_2026.csv')

# Df plotagem dos acummulados: vai ser de 01/04 do ano q compro a carteira até 29/03 do ano da frente
df_acumulados_ativos_2015 = df_ativos.loc['2015-04-01':'2016-03-31']
df_acumulados_ativos_2016 = df_ativos.loc['2016-04-01':'2017-03-31']
df_acumulados_ativos_2017 = df_ativos.loc['2017-04-01':'2018-03-31']
df_acumulados_ativos_2018 = df_ativos.loc['2018-04-01':'2019-03-31']
df_acumulados_ativos_2019 = df_ativos.loc['2019-04-01':'2020-03-31']
df_acumulados_ativos_2020 = df_ativos.loc['2020-04-01':'2021-03-31']
df_acumulados_ativos_2021 = df_ativos.loc['2021-04-01':'2022-03-31']
df_acumulados_ativos_2022 = df_ativos.loc['2022-04-01':'2023-03-31']
df_acumulados_ativos_2023 = df_ativos.loc['2023-04-01':'2024-03-31']   
df_acumulados_ativos_2024 = df_ativos.loc['2024-04-01':'2025-03-31']
df_acumulados_ativos_2025 = df_ativos.loc['2025-04-01':'2026-03-31']

df_acumulados_ativos_2015.to_csv('retornos_acumulados_plotagem/ativo/df_acumulados_ativos_2015.csv')
df_acumulados_ativos_2016.to_csv('retornos_acumulados_plotagem/ativo/df_acumulados_ativos_2016.csv')
df_acumulados_ativos_2017.to_csv('retornos_acumulados_plotagem/ativo/df_acumulados_ativos_2017.csv')
df_acumulados_ativos_2018.to_csv('retornos_acumulados_plotagem/ativo/df_acumulados_ativos_2018.csv')
df_acumulados_ativos_2019.to_csv('retornos_acumulados_plotagem/ativo/df_acumulados_ativos_2019.csv')
df_acumulados_ativos_2020.to_csv('retornos_acumulados_plotagem/ativo/df_acumulados_ativos_2020.csv')
df_acumulados_ativos_2021.to_csv('retornos_acumulados_plotagem/ativo/df_acumulados_ativos_2021.csv')
df_acumulados_ativos_2022.to_csv('retornos_acumulados_plotagem/ativo/df_acumulados_ativos_2022.csv')
df_acumulados_ativos_2023.to_csv('retornos_acumulados_plotagem/ativo/df_acumulados_ativos_2023.csv')
df_acumulados_ativos_2024.to_csv('retornos_acumulados_plotagem/ativo/df_acumulados_ativos_2024.csv')
df_acumulados_ativos_2025.to_csv('retornos_acumulados_plotagem/ativo/df_acumulados_ativos_2025.csv')

#ibov
# df_ibov_2015 = df_ibov[df_ibov.index.str.startswith('2015')]
# df_ibov_2016 = df_ibov[df_ibov.index.str.startswith('2016')]
# df_ibov_2017 = df_ibov[df_ibov.index.str.startswith('2017')]
# df_ibov_2018 = df_ibov[df_ibov.index.str.startswith('2018')]
# df_ibov_2019 = df_ibov[df_ibov.index.str.startswith('2019')]
# df_ibov_2020 = df_ibov[df_ibov.index.str.startswith('2020')]
# df_ibov_2021 = df_ibov[df_ibov.index.str.startswith('2021')]
# df_ibov_2022 = df_ibov[df_ibov.index.str.startswith('2022')]
# df_ibov_2023 = df_ibov[df_ibov.index.str.startswith('2023')]
# df_ibov_2024 = df_ibov[df_ibov.index.str.startswith('2024')]
# df_ibov_2025 = df_ibov[df_ibov.index.str.startswith('2025')]
# df_ibov_2026 = df_ibov[df_ibov.index.str.startswith('2026')]

df_ibov_retornos_2015 = df_ibov.loc['2015-10-01':'2016-03-31']
df_ibov_retornos_2016 = df_ibov.loc['2016-10-01':'2017-03-31']
df_ibov_retornos_2017 = df_ibov.loc['2017-10-01':'2018-03-31']
df_ibov_retornos_2018 = df_ibov.loc['2018-10-01':'2019-03-31']
df_ibov_retornos_2019 = df_ibov.loc['2019-10-01':'2020-03-31']
df_ibov_retornos_2020 = df_ibov.loc['2020-10-01':'2021-03-31']
df_ibov_retornos_2021 = df_ibov.loc['2021-10-01':'2022-03-31']
df_ibov_retornos_2022 = df_ibov.loc['2022-10-01':'2023-03-31']
df_ibov_retornos_2023 = df_ibov.loc['2023-10-01':'2024-03-31']   
df_ibov_retornos_2024 = df_ibov.loc['2024-10-01':'2025-03-31']
df_ibov_retornos_2025 = df_ibov.loc['2025-10-01':'2026-03-31']

df_ibov_retornos_2015.to_csv('retornos_ibov_otimizacao/df_ibov_retornos_2015.csv')
df_ibov_retornos_2016.to_csv('retornos_ibov_otimizacao/df_ibov_retornos_2016.csv')
df_ibov_retornos_2017.to_csv('retornos_ibov_otimizacao/df_ibov_retornos_2017.csv')
df_ibov_retornos_2018.to_csv('retornos_ibov_otimizacao/df_ibov_retornos_2018.csv')
df_ibov_retornos_2019.to_csv('retornos_ibov_otimizacao/df_ibov_retornos_2019.csv')
df_ibov_retornos_2020.to_csv('retornos_ibov_otimizacao/df_ibov_retornos_2020.csv')
df_ibov_retornos_2021.to_csv('retornos_ibov_otimizacao/df_ibov_retornos_2021.csv')
df_ibov_retornos_2022.to_csv('retornos_ibov_otimizacao/df_ibov_retornos_2022.csv')
df_ibov_retornos_2023.to_csv('retornos_ibov_otimizacao/df_ibov_retornos_2023.csv')
df_ibov_retornos_2024.to_csv('retornos_ibov_otimizacao/df_ibov_retornos_2024.csv')
df_ibov_retornos_2025.to_csv('retornos_ibov_otimizacao/df_ibov_retornos_2025.csv')

df_ibov_2015 = df_ibov.loc['2015-04-01':'2016-03-31']
df_ibov_2016 = df_ibov.loc['2016-04-01':'2017-03-31']
df_ibov_2017 = df_ibov.loc['2017-04-01':'2018-03-31']
df_ibov_2018 = df_ibov.loc['2018-04-01':'2019-03-31']
df_ibov_2019 = df_ibov.loc['2019-04-01':'2020-03-31']
df_ibov_2020 = df_ibov.loc['2020-04-01':'2021-03-31']
df_ibov_2021 = df_ibov.loc['2021-04-01':'2022-03-31']
df_ibov_2022 = df_ibov.loc['2022-04-01':'2023-03-31']
df_ibov_2023 = df_ibov.loc['2023-04-01':'2024-03-31']
df_ibov_2024 = df_ibov.loc['2024-04-01':'2025-03-31']
df_ibov_2025 = df_ibov.loc['2025-04-01':'2026-03-31']
# df_ibov_2026 = df_ibov.loc['2026-04-01':'2027-03-31']



df_ibov_2015.to_csv('retornos_acumulados_plotagem/ibov/df_ibov_2015.csv')
df_ibov_2016.to_csv('retornos_acumulados_plotagem/ibov/df_ibov_2016.csv')
df_ibov_2017.to_csv('retornos_acumulados_plotagem/ibov/df_ibov_2017.csv')
df_ibov_2018.to_csv('retornos_acumulados_plotagem/ibov/df_ibov_2018.csv')
df_ibov_2019.to_csv('retornos_acumulados_plotagem/ibov/df_ibov_2019.csv')
df_ibov_2020.to_csv('retornos_acumulados_plotagem/ibov/df_ibov_2020.csv')
df_ibov_2021.to_csv('retornos_acumulados_plotagem/ibov/df_ibov_2021.csv')
df_ibov_2022.to_csv('retornos_acumulados_plotagem/ibov/df_ibov_2022.csv')
df_ibov_2023.to_csv('retornos_acumulados_plotagem/ibov/df_ibov_2023.csv')
df_ibov_2024.to_csv('retornos_acumulados_plotagem/ibov/df_ibov_2024.csv')
df_ibov_2025.to_csv('retornos_acumulados_plotagem/ibov/df_ibov_2025.csv')
# df_ibov_2026.to_csv('retornos_ibov/df_ibov_2026.csv')

## EXCESSO DOS ANOS e SIGMA (MAtriz de covariancia)

##### EXCESSO PARA TODOS

In [45]:
lista_macro = []

for filename in os.listdir(path='../../base_dados/brapi/macro'):
    
    # 2. Reconstruct the full absolute or relative path to the file
    full_path = os.path.join('../../base_dados/brapi/macro/',filename)
    
    # 3. Check if the current item is actually a file (and not a subfolder)
    # if os.path.isfile(full_path):
        
    #     # 4. Open and process the file safely
    #     with open(full_path, "r", encoding="utf-8") as file:
    #         content = file.read()
    #         print(f"--- Content of {filename} ---")
    #         print(pd.read_csv(full_path))
    dicio = {
        'macro':filename,
        'data':pd.read_csv(full_path)
    }

    lista_macro.append(dicio)
lista_macro[0]['data'].set_index('date', inplace=True)
selic_d = lista_macro[0]['data']
selic_d.to_csv('selic/selic_diario.csv')